# SBD Model Demo (Request 9)

Демонстрация включает:
- сквозной tracing (`trace_id`, `span_id`, `parent_span_id`);
- сертификацию прошивки (`VendorUAS -> Regulator`);
- регистрацию БАС с форматом `UAS-<TYPE>-<VENDOR_CODE>-<NUMBER>`;
- закупку БАС оператором и приписку к `DronePort`;
- основной e2e-сценарий заказа.

## Инициализация окружения

Импортируем runtime-функции из `sbd_demo.py` и поднимаем все процессы сущностей.

In [ ]:
import sys
import time
import multiprocessing as mp
from pathlib import Path
from typing import Any, Dict, List, Optional
import queue as pyqueue


def _find_repo_root(start: Path) -> Path:
    root = start
    for _ in range(8):
        if (root / "broker").exists() and (root / "systems").exists():
            return root
        if root.parent == root:
            break
        root = root.parent
    return start


repo_root = _find_repo_root(Path.cwd())
sbd_demo_code_dir = repo_root / "demos" / "sbd-model-simple-demo" / "sbd-model-demo-code"
sys.path.insert(0, str(sbd_demo_code_dir))

from actions import (
    PLACE_ORDER,
    REQUEST_FIRMWARE_CERTIFICATION,
    REQUEST_UAS_PURCHASE,
    REQUEST_UAS_REGISTRATION,
)
from sbd_demo import (
    _launch,
    _send_rpc_from_orchestrator,
    _shutdown,
    _wait_for_rpc_main,    
)

from world_state import build_world

world = build_world()
runtime = _launch(world)
out_q = runtime["orchestrator_reply_queue"]
out_name = runtime["orchestrator_reply_name"]
in_q = runtime["broker_in_queue"]

[broker] start pid=3704294 log=/home/user/projects/sbd-drones-economics/sbd-drones-economics-ai/demos/sbd-model-simple-demo/simulation.log


[broker] orchestrator->vendor_uas action=request_firmware_certification type=request corr=98973a7950584e72b703e6a87d27b84a trace=954776a48d434f9e908e0b7f91f1fc61 span=e4db818986e64a24 parent_span=None payload={'firmware_version': '2.1.0', 'vendor_code': 'SBD', 'uas_type': 'AGRO', 'artifacts': ['report.pdf', 'tests.json']}
[broker] vendor_uas->regulator action=request_firmware_certification type=request corr=8f610b9b20a648ae8e36ba5fa2fd8ed8 trace=954776a48d434f9e908e0b7f91f1fc61 span=940ffd2573de4fe3 parent_span=e4db818986e64a24 payload={'firmware_version': '2.1.0', 'vendor_code': 'SBD', 'uas_type': 'AGRO', 'artifacts': ['report.pdf', 'tests.json']}
[broker] regulator->reply_vendor_uas action=request_firmware_certification type=response corr=8f610b9b20a648ae8e36ba5fa2fd8ed8 trace=954776a48d434f9e908e0b7f91f1fc61 span=48e22390ee11435d parent_span=940ffd2573de4fe3 payload={'status': 'ok', 'firmware_certification_result': {'approved': True, 'certificate_id': 'CERT-AGRO-SBD-2_1_0', 'firmwar

[broker] vendor_uas->reply_orchestrator action=request_firmware_certification type=response corr=98973a7950584e72b703e6a87d27b84a trace=954776a48d434f9e908e0b7f91f1fc61 span=5cc0728e085e4e94 parent_span=e4db818986e64a24 payload={'status': 'ok', 'firmware_certification_result': {'approved': True, 'certificate_id': 'CERT-AGRO-SBD-2_1_0', 'firmware_version': '2.1.0'}}
[broker] orchestrator->vendor_uas action=request_uas_registration type=request corr=51d6c3a96438408eb674ab70e9a03909 trace=de2ff0986a9e49e49d1284689dcee313 span=36b7627353f848a5 parent_span=None payload={'firmware_version': '2.1.0', 'vendor_code': 'SBD', 'uas_type': 'AGRO', 'imei': '356938035643809'}
[broker] vendor_uas->regulator action=request_uas_registration type=request corr=83dfc187bb724664be5f60b63be0fdd8 trace=de2ff0986a9e49e49d1284689dcee313 span=b156d3bb497d4ce9 parent_span=36b7627353f848a5 payload={'uas_type': 'AGRO', 'vendor_code': 'SBD', 'imei': '356938035643809', 'firmware_version': '2.1.0'}
[broker] regulator-

## Сценарий 1: Сертификация прошивки

Проверяем, что цепочка проходит через `VendorUAS` и `Regulator`, а в ответе приходит сертификат.

In [4]:
cert_corr, cert_trace = _send_rpc_from_orchestrator(
    broker_in_queue=in_q,
    reply_to=out_name,
    receiver="vendor_uas",
    action=REQUEST_FIRMWARE_CERTIFICATION,
    payload={
        "firmware_version": "2.1.0",
        "vendor_code": "SBD",
        "uas_type": "AGRO",
        "artifacts": ["report.pdf", "tests.json"],
    },
)
cert_msg = _wait_for_rpc_main(out_q, correlation_id=cert_corr, expected_sender="vendor_uas", timeout_s=60.0)
assert cert_msg["trace_id"] == cert_trace
firmware_cert = cert_msg["payload"]["firmware_certification_result"]
assert firmware_cert["approved"] is True
firmware_cert

{'approved': True,
 'certificate_id': 'CERT-AGRO-SBD-2_1_0',
 'firmware_version': '2.1.0'}

## Сценарий 2: Регистрация БАС

Проверяем формат `registered_uas_id` и трассировку.

In [5]:
import re

reg_corr, reg_trace = _send_rpc_from_orchestrator(
    broker_in_queue=in_q,
    reply_to=out_name,
    receiver="vendor_uas",
    action=REQUEST_UAS_REGISTRATION,
    payload={
        "firmware_version": "2.1.0",
        "vendor_code": "SBD",
        "uas_type": "AGRO",
        "imei": "356938035643809",
    },
)
reg_msg = _wait_for_rpc_main(out_q, correlation_id=reg_corr, expected_sender="vendor_uas", timeout_s=60.0)
assert reg_msg["trace_id"] == reg_trace
registered_uas_id = reg_msg["payload"]["registered_uas_id"]
assert re.fullmatch(r"UAS-[A-Z0-9_]+-[A-Z0-9_]+-\d{6}", registered_uas_id)
registered_uas_id

'UAS-AGRO-SBD-000001'

## Сценарий 3: Закупка БАС Operator и приписка к DronePort

Оператор закупает 2 новые БАС у `VendorUAS`, затем приписывает их к `droneport_B`.

In [6]:
purchase_corr, purchase_trace = _send_rpc_from_orchestrator(
    broker_in_queue=in_q,
    reply_to=out_name,
    receiver="operator_1",
    action=REQUEST_UAS_PURCHASE,
    payload={
        "quantity": 2,
        "target_droneport_id": "droneport_B",
        "vendor_code": "SBD",
        "uas_type": "AGRO",
        "firmware_version": "2.1.0",
        "model_id": "agro-model-new",
        "supported_task_types": ["agro"],
        "base_cost": 9800.0,
        "imeis": ["356938035643810", "356938035643811"],
    },
)
purchase_msg = _wait_for_rpc_main(out_q, correlation_id=purchase_corr, expected_sender="operator_1", timeout_s=60.0)
assert purchase_msg["trace_id"] == purchase_trace
purchase_payload = purchase_msg["payload"]
for uas_id in purchase_payload["uas_purchase_result"]["registered_uas_ids"]:
    assert re.fullmatch(r"UAS-[A-Z0-9_]+-[A-Z0-9_]+-\d{6}", uas_id)
assert purchase_payload["attachment"]["droneport_id"] == "droneport_B"
assert purchase_payload["attachment"]["assigned_count"] == 2
purchase_payload

{'status': 'ok',
 'uas_purchase_result': {'registered_uas_ids': ['UAS-AGRO-SBD-000002',
   'UAS-AGRO-SBD-000003'],
  'quantity': 2},
 'attachment': {'droneport_id': 'droneport_B', 'assigned_count': 2},
 'purchased_uas_items': [{'uas_id': 'UAS-AGRO-SBD-000002',
   'model_id': 'agro-model-new',
   'supported_task_types': ['agro'],
   'base_cost': 9800.0},
  {'uas_id': 'UAS-AGRO-SBD-000003',
   'model_id': 'agro-model-new',
   'supported_task_types': ['agro'],
   'base_cost': 9800.0}]}

## Сценарий 4: Базовый end-to-end заказ

После новых сценариев запускаем основной поток заказа и проверяем результат посадки.

In [7]:
order_corr, order_trace = _send_rpc_from_orchestrator(
    broker_in_queue=in_q,
    reply_to=out_name,
    receiver="customer",
    action=PLACE_ORDER,
    payload={
        "order": {
            "id": "ORDER-DEMO-001",
            "scenario_type": "agro",
            "destination": {"lat": 55.75, "lon": 37.61},
            "return_port": "droneport_B",
            "coverage": {
                "min_payload": 3.5,
                "min_range": 1.0,
                "min_battery": 0.8,
            },
        },
        "scenario_security_goals": ["SG_ID_AUTH_001", "SG_ID_SAF_002"],
        "max_price": 999999.0,
    },
)
final_msg = _wait_for_rpc_main(out_q, correlation_id=order_corr, expected_sender="customer", timeout_s=180.0)
assert final_msg["trace_id"] == order_trace
final_payload = final_msg["payload"]
assert final_payload["status"] == "ok"
assert final_payload["order_execution_completed"]["landing_coordinates"] == [55.76, 37.62]
final_payload

{'status': 'ok',
 'order_execution_completed': {'status': 'ok',
  'landing_coordinates': [55.76, 37.62],
  'mission_id': 'mission-b2fea241',
  'uas_id': 'uas_A_01'},
 'selected_operator': 'operator_1',
 'order_security_goals': ['SG_ID_AUTH_001',
  'SG_ID_SAF_002',
  'SG_ID_CONSTR_010'],
 'chosen_proposal': {'proposal_id': 'prop-operator_1-a511a7',
  'price': 11000.0,
  'margin_percent': 10.0,
  'applied_security_goals': ['SG_ID_AUTH_001',
   'SG_ID_SAF_002',
   'SG_ID_CONSTR_010']}}

## Завершение

Останавливаем процессы и выводим агрегированный результат.

In [8]:
_shutdown(runtime)
artifacts = {
    "firmware_cert": firmware_cert,
    "registered_uas_id": registered_uas_id,
    "purchased_uas_ids": purchase_payload["uas_purchase_result"]["registered_uas_ids"],
    "final_order_result": final_payload,
}
artifacts

{'firmware_cert': {'approved': True,
  'certificate_id': 'CERT-AGRO-SBD-2_1_0',
  'firmware_version': '2.1.0'},
 'registered_uas_id': 'UAS-AGRO-SBD-000001',
 'purchased_uas_ids': ['UAS-AGRO-SBD-000002', 'UAS-AGRO-SBD-000003'],
 'final_order_result': {'status': 'ok',
  'order_execution_completed': {'status': 'ok',
   'landing_coordinates': [55.76, 37.62],
   'mission_id': 'mission-b2fea241',
   'uas_id': 'uas_A_01'},
  'selected_operator': 'operator_1',
  'order_security_goals': ['SG_ID_AUTH_001',
   'SG_ID_SAF_002',
   'SG_ID_CONSTR_010'],
  'chosen_proposal': {'proposal_id': 'prop-operator_1-a511a7',
   'price': 11000.0,
   'margin_percent': 10.0,
   'applied_security_goals': ['SG_ID_AUTH_001',
    'SG_ID_SAF_002',
    'SG_ID_CONSTR_010']}}}